In [1]:
import pandas as pd
import torch
from train_utils import ItemDataset, QueriesDataset, get_device, evaluate
from models import TwoTowerHF
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch.nn.functional as F


seed = 42

d:\projects\avito_retr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Загрузка трансформера

In [ ]:
device = get_device()
transformer = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device=device,
    cache_folder="models/hf"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8288.89it/s]


### Train

In [3]:
queries_df_hf = pd.read_parquet('preprocessed/train_queries_hf.parquet')
items_df_hf= pd.read_parquet('preprocessed/train_items_hf.parquet')

In [4]:
g = torch.Generator()
g.manual_seed(seed)

item_dataset_hf = ItemDataset(transformer, items_df=items_df_hf)

train_df, val_df = train_test_split(queries_df_hf, test_size=0.1, random_state=seed)
train_dataset = QueriesDataset(queries_df=train_df, item_dataset=item_dataset_hf)
val_dataset = QueriesDataset(queries_df=val_df, item_dataset=item_dataset_hf, mode='train')

Use SentenceTransformer


Batches: 100%|██████████| 5388/5388 [02:38<00:00, 34.06it/s]


Use SentenceTransformer


Batches: 100%|██████████| 6999/6999 [02:40<00:00, 43.68it/s]


Use SentenceTransformer


Batches: 100%|██████████| 778/778 [00:18<00:00, 42.42it/s]


In [5]:
model = TwoTowerHF(context_proj=True)
train_dataloader = DataLoader(train_dataset, 
                              num_workers=0, 
                              shuffle=True, 
                              batch_size=1024,
                              generator=g)

In [6]:
device = get_device()
model.to(device)
optimizer = torch.optim.Adam(model.parameters())

temperature = 0.05
n_epochs = 20
batches_per_epoch = 300
best_val_recall = -torch.inf
max_bad_epochs = 3

for epoch in range(n_epochs):
    model.train()   
    batch_iter = iter(train_dataloader)
    pbar = tqdm(range(batches_per_epoch))
    loss_sum = 0
    for batch_idx in pbar:
        q_emb, i_emb, context, item_ids, _, _ = [tensor.to(device) for tensor in next(batch_iter)]
        
        q, i = model((q_emb, i_emb, context))
        logits = (q @ i.T) / temperature
        
        same_item = item_ids[:, None] == item_ids[None, :]
        diag = torch.arange(logits.size(0), device=device)
        same_item[diag, diag] = False

        logits[same_item] = -torch.inf
        targets = torch.arange(logits.size(0), device=device)
        loss = F.cross_entropy(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        pbar.set_description(f"Epoch {epoch} --- loss: {loss_sum / (batch_idx + 1)} | ")

    # recall = evaluate(queries_dataset=val_dataset, items_dataset=item_dataset, model=model, device=device, k=100)
    # print(f'Recall@100: {recall}')
    # recall = evaluate(queries_dataset=val_dataset, items_dataset=item_dataset_hf, model=model, device=device, k=50, filters=False)
    # print(recall)
    # if best_val_recall > recall['Recall@50']:
    #     max_bad_epochs -= 1
    # else:
    #     best_val_recall = recall['Recall@50']
    #     torch.save(model.state_dict(), "models/hf_towers/two_tower_hf.pth")
    # if max_bad_epochs == 0:
    #     break

Epoch 0 --- loss: 6.869292491277059 | : 100%|██████████| 300/300 [00:14<00:00, 20.19it/s] 
Epoch 1 --- loss: 4.557318449020386 | : 100%|██████████| 300/300 [00:13<00:00, 22.90it/s] 
Epoch 2 --- loss: 3.064601104259491 | : 100%|██████████| 300/300 [00:12<00:00, 23.22it/s] 
Epoch 3 --- loss: 2.859205967585246 | : 100%|██████████| 300/300 [00:13<00:00, 23.00it/s] 
Epoch 4 --- loss: 2.746303930282593 | : 100%|██████████| 300/300 [00:12<00:00, 23.36it/s] 
Epoch 5 --- loss: 2.679841523170471 | : 100%|██████████| 300/300 [00:12<00:00, 23.52it/s] 
Epoch 6 --- loss: 2.635680156548818 | : 100%|██████████| 300/300 [00:13<00:00, 22.95it/s] 
Epoch 7 --- loss: 2.5906051365534464 | : 100%|██████████| 300/300 [00:12<00:00, 23.22it/s]
Epoch 8 --- loss: 2.56525058110555 | : 100%|██████████| 300/300 [00:12<00:00, 23.57it/s]  
Epoch 9 --- loss: 2.5838463274637857 | : 100%|██████████| 300/300 [00:12<00:00, 23.18it/s]
Epoch 10 --- loss: 2.523072911898295 | : 100%|██████████| 300/300 [00:12<00:00, 23.18it/s]

In [7]:
recall = evaluate(queries_dataset=val_dataset, items_dataset=item_dataset_hf, model=model, device=device, k=50, filters=False)
recall_filtered = evaluate(queries_dataset=val_dataset, items_dataset=item_dataset_hf, model=model, device=device, k=50, filters=True)

Queries: 100%|██████████| 778/778 [00:31<00:00, 24.97it/s]


In [8]:
print(recall)
print(recall_filtered)

{'Recall@50': 0.12335235492686063}
{'Recall@50': 0.7398328243047742}


e5-small:
1. {'Recall@50': 0.12238787976209613}
2. {'Recall@50': 0.7354123131329369}

### gen ans

In [2]:
queries_df_test = pd.read_parquet('preprocessed/benchmark_queries_preprocessed_hf.parquet')
items_df_test = pd.read_parquet('preprocessed/benchmark_items_preprocessed_hf.parquet')

In [ ]:
device = get_device()
transformer = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device=device,
    cache_folder="models/hf"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 529.07it/s]


In [4]:
g = torch.Generator()
g.manual_seed(seed)

item_dataset_test = ItemDataset(transformer, items_df=items_df_test)
test_dataset = QueriesDataset(queries_df=queries_df_test, item_dataset=item_dataset_test, mode='test')

model = TwoTowerHF()
model.load_state_dict(torch.load("models/hf_towers/two_tower_hf.pth"))
model.to(device)

Use SentenceTransformer


Batches: 100%|██████████| 2957/2957 [01:26<00:00, 34.36it/s]


Use SentenceTransformer


Batches: 100%|██████████| 39/39 [00:00<00:00, 41.53it/s]


TwoTowerHF(
  (q_tower): QTowerHF(
    (dense): Sequential(
      (0): Linear(in_features=384, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
      (3): ReLU()
    )
  )
  (i_tower): ITowerHF(
    (context_proj): Sequential(
      (0): Linear(in_features=5, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=128, bias=True)
      (3): ReLU()
    )
    (dense): Sequential(
      (0): Linear(in_features=512, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
      (3): ReLU()
    )
  )
)

In [5]:
ans = evaluate(queries_dataset=test_dataset, items_dataset=item_dataset_test, model=model, device=device, k=50, filters=True)

Queries: 100%|██████████| 39/39 [00:01<00:00, 36.72it/s]


In [6]:
answer = pd.DataFrame({
    'query_id': queries_df_test['query_id'],                                   # строки
    'answer': [' '.join(top50) for top50 in item_dataset_test.item_ids[torch.cat(ans, dim=0)]],     # строки
})
answer.to_csv('ans/answer_hf.csv', index=False)

Recall@50: 0.644332